# NASA CMAPSS FD001 — Exploratory Data Analysis
**Project:** Predictive Equipment Failure | PyTorch LSTM + Vertex AI  
**Dataset:** NASA CMAPSS FD001 — Turbofan Engine Degradation Simulation  
**Output:** `data/eda_summary.json` — feeds directly into Phase 3 feature engineering

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Allow importing src/ from the repo root
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.loader import CMAPSSLoader

plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})
sns.set_theme(style="whitegrid", palette="muted")

RUL_CAP = 125  # Piecewise linear cap — standard for FD001

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────────────
loader = CMAPSSLoader(data_dir=str(ROOT / "data" / "raw"))
# Uncomment the line below the first time you run this notebook:
# loader.download()

train_df, test_df, rul_series = loader.load_all()

---
## Section 1 — Dataset Overview

In [ ]:
print("=" * 55)
print("TRAINING SET")
print(f"  Shape          : {train_df.shape}")
print(f"  Engines        : {train_df['engine_id'].nunique()}")
print(f"  Cycle range    : {train_df['cycle'].min()} – {train_df['cycle'].max()}")
print(f"  Sensor cols    : {len([c for c in train_df.columns if c.startswith('sensor_')])}")
print(f"  Missing values : {train_df.isnull().sum().sum()}")
print()
print("TEST SET")
print(f"  Shape          : {test_df.shape}")
print(f"  Engines        : {test_df['engine_id'].nunique()}")
print()
print("RUL GROUND TRUTH")
print(f"  Entries        : {len(rul_series)}")
print(f"  Range          : {rul_series.min():.0f} – {rul_series.max():.0f} cycles")
print("=" * 55)

In [ ]:
train_df.head(3)

In [ ]:
train_df.dtypes

---
## Section 2 — RUL Distribution Analysis

In [ ]:
# Compute RUL for each cycle in training set
max_cycle = train_df.groupby("engine_id")["cycle"].max().rename("max_cycle")
train_df = train_df.join(max_cycle, on="engine_id")
train_df["rul_raw"] = train_df["max_cycle"] - train_df["cycle"]
train_df["rul"] = train_df["rul_raw"].clip(upper=RUL_CAP)  # Piecewise linear cap
train_df.drop(columns=["max_cycle"], inplace=True)

print(f"RUL stats (capped at {RUL_CAP}):")
print(train_df["rul"].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# RUL distribution histogram
sns.histplot(train_df["rul"], bins=40, ax=axes[0], color="steelblue")
axes[0].set_title(f"RUL Distribution (cap={RUL_CAP})")
axes[0].set_xlabel("Remaining Useful Life (cycles)")
axes[0].set_ylabel("Count")

# RUL over cycles for 5 sample engines
sample_engines = train_df["engine_id"].unique()[:5]
for eid in sample_engines:
    subset = train_df[train_df["engine_id"] == eid]
    axes[1].plot(subset["cycle"], subset["rul"], label=f"Engine {eid}")
axes[1].set_title("Capped RUL over Cycles (5 engines)")
axes[1].set_xlabel("Cycle")
axes[1].set_ylabel("RUL (capped)")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Section 3 — Sensor Analysis

In [ ]:
sensor_cols = [f"sensor_{i}" for i in range(1, 22)]

# Identify zero-variance sensors
sensor_std = train_df[sensor_cols].std()
zero_var_sensors = sensor_std[sensor_std == 0].index.tolist()
print(f"Zero-variance sensors (will be dropped): {zero_var_sensors}")
print(f"Expected for FD001: sensor_1, sensor_5, sensor_6, sensor_10, sensor_16, sensor_18, sensor_19")

In [ ]:
# Plot all 21 sensors for one engine
engine_1 = train_df[train_df["engine_id"] == 1]

fig, axes = plt.subplots(7, 3, figsize=(16, 20))
axes = axes.flatten()

for idx, sensor in enumerate(sensor_cols):
    color = "tomato" if sensor in zero_var_sensors else "steelblue"
    axes[idx].plot(engine_1["cycle"], engine_1[sensor], color=color, linewidth=0.8)
    axes[idx].set_title(sensor + (" [DROP]" if sensor in zero_var_sensors else ""), fontsize=9)
    axes[idx].set_xlabel("Cycle", fontsize=7)
    axes[idx].tick_params(labelsize=7)

plt.suptitle("All 21 Sensors — Engine 1 (red = zero variance, will be dropped)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Retained sensors after dropping zero-variance
active_sensors = [s for s in sensor_cols if s not in zero_var_sensors]
print(f"Retained sensors ({len(active_sensors)}): {active_sensors}")

---
## Section 4 — Correlation Analysis

In [ ]:
# Sensor-to-RUL correlations (use active sensors only)
corr_with_rul = (
    train_df[active_sensors + ["rul"]]
    .corr()["rul"]
    .drop("rul")
    .abs()
    .sort_values(ascending=False)
)

print("Top sensors by |correlation| with RUL:")
print(corr_with_rul.round(3))

top_sensors = corr_with_rul.head(8).index.tolist()
print(f"\nTop 8 sensors: {top_sensors}")

In [ ]:
# Correlation heatmap — active sensors vs RUL
corr_matrix = train_df[active_sensors + ["rul"]].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    annot=False,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    ax=ax,
    linewidths=0.3,
)
ax.set_title("Sensor Correlation Heatmap (active sensors + RUL)")
plt.tight_layout()
plt.show()

---
## Section 5 — Operational Settings

In [ ]:
op_cols = ["op_setting_1", "op_setting_2", "op_setting_3"]
op_std = train_df[op_cols].std()
print("Operational settings standard deviation:")
print(op_std.round(6))
print()
print("FD001 uses a single operating condition.")
print("Operational settings are effectively constant — confirming single fault mode.")
print()
print("Note for README: 'FD001 chosen for single fault mode clarity'")

---
## Section 6 — Summary Statistics

In [ ]:
stats = train_df[active_sensors].agg(["min", "max", "mean", "std"]).T.round(3)

# Flag outliers: sensors where any value is > 3 std from mean
sensor_means = train_df[active_sensors].mean()
sensor_stds = train_df[active_sensors].std()
outlier_flags = ((train_df[active_sensors] - sensor_means).abs() > 3 * sensor_stds).any()
stats["has_outliers"] = outlier_flags

print("Summary statistics per active sensor:")
print(stats.to_string())

---
## Save EDA Summary

In [ ]:
eda_summary = {
    "rul_cap": RUL_CAP,
    "n_train_engines": int(train_df["engine_id"].nunique()),
    "n_test_engines": int(test_df["engine_id"].nunique()),
    "max_rul": float(train_df["rul"].max()),
    "mean_rul": round(float(train_df["rul"].mean()), 2),
    "drop_sensors": zero_var_sensors,
    "active_sensors": active_sensors,
    "n_features": len(active_sensors),
    "top_sensors": top_sensors,
    "corr_with_rul": corr_with_rul.round(4).to_dict(),
    "op_settings_constant": bool((op_std < 0.01).all()),
}

output_path = ROOT / "data" / "eda_summary.json"
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, "w") as f:
    json.dump(eda_summary, f, indent=2)

print(f"EDA summary saved to: {output_path}")
print(json.dumps({k: v for k, v in eda_summary.items() if k != "corr_with_rul"}, indent=2))

---
## Phase 2.3 — Upload Raw Data to GCS

Run in **Google Cloud Shell** after downloading data:

```bash
export GCS_BUCKET_NAME=predictive-maintenance-artifacts
gsutil -m cp data/raw/*.txt gs://$GCS_BUCKET_NAME/data/raw/
```

---
✅ **EDA complete.** Confirm `data/eda_summary.json` exists, then proceed to Phase 3.